# 04.3 · Modelagem — o cotovelo com uma única variável: `ano_reforma`

**CRISP-DM fase 4 · modelagem** · Meridiano Imobiliária

---

O [04.1](04-1_modelagem_cotovelo.ipynb) e o [04.2](04-2_modelagem_cotovelo.ipynb) agruparam os
imóveis por todas as variáveis e chegaram a silhuetas entre 0,10 e 0,13. A meta aqui é outra:
**silhueta entre 0,7 e 0,9**. A busca sobre combinações de variáveis feita antes deste notebook
mostrou que, sem cair em grupos triviais (os valores distintos de uma variável discreta, ou o
"tem / não tem" de uma área quase sempre zerada), a única variável que chega a essa faixa é o
**ano da última reforma** (`ano_reforma`).

Este notebook roda os mesmos oito algoritmos e o mesmo cotovelo ao vivo sobre **essa única
coluna**. Ele deixa explícito, desde o início, o que isso significa:

* **os grupos serão faixas de ano.** Em uma dimensão, todo agrupamento é um conjunto de cortes
  na reta — "reformados até 1980" e "reformados depois". A silhueta sobe porque não há outra
  variável para embaralhar a distância, e não porque os imóveis se separem melhor em blocos;
* **as silhuetas não se comparam com as do 04.1 e do 04.2.** A régua é outra: lá, a distância
  somava ~80 a ~200 colunas; aqui, é a diferença entre dois anos.

O bloco 7 mostra os grupos como faixas, descreve cada uma pelas demais variáveis e compara
com um corte simples por quantis, sem algoritmo nenhum.

Cada algoritmo é treinado **para cada número de grupos**, e o leitor acompanha, em tempo de
execução, duas curvas lado a lado:

* **o peso** — o custo que o algoritmo minimiza (soma das distâncias, inércia,
  −log-verossimilhança). Ele **sempre cai** quando *k* aumenta: mais grupos, cada ponto mais
  perto do seu centro. O **cotovelo** é o ponto a partir do qual um grupo a mais deixa de
  comprar redução relevante de peso;
* **a silhueta** — a métrica de acompanhamento comum a todos. Vai de −1 a 1 e mede, para cada
  imóvel, se ele está mais perto do próprio grupo do que do grupo vizinho. Ao contrário do
  peso, ela **não melhora sozinha** com *k*, e por isso serve de contraprova ao cotovelo.

Os algoritmos treinados:

| # | Algoritmo | O que é *k* | Peso (o que cai com *k*) | Silhueta medida em |
|---|---|---|---|---|
| 1 | K-Means · euclidiana | número de centroides | inércia (soma dos quadrados) | distância euclidiana |
| 2 | K-Means · Manhattan | número de centros | soma das distâncias L1 | distância Manhattan |
| 3 | K-Means · Mahalanobis | número de centroides | soma dos quadrados de Mahalanobis | distância de Mahalanobis |
| 4 | K-Medoids | número de medoides | soma das distâncias ao medoide | distância euclidiana |
| 5 | Mistura gaussiana (GMM) | número de componentes | −log-verossimilhança | distância euclidiana |
| 6 | Hierárquica (Ward) | altura do corte no dendrograma | soma dos quadrados intra-grupo | distância euclidiana |
| 7 | Mean Shift | **emerge** da largura de banda | soma dos quadrados intra-grupo | distância euclidiana |
| 8 | DBSCAN | **emerge** do raio `eps` | soma dos quadrados, ruído sem grupo | distância euclidiana, sem ruído |

Em Mean Shift e DBSCAN o número de grupos não é parâmetro. Neles a varredura percorre o
parâmetro que **controla** *k* (do maior para o menor), e cada ponto do gráfico traz o *k*
que resultou.

Cada silhueta é medida na distância que o próprio algoritmo usa. Para comparar algoritmos
entre si, a tabela final traz também a **silhueta comum**: a euclidiana, no mesmo espaço,
para todos.

## 0 · Ambiente

In [ ]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.spatial.distance import cdist, pdist, squareform
from sklearn.cluster import DBSCAN, KMeans, MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.model import densidade as den
from src.model import hierarquica as hier
from src.model import particional as part
from src.utils import io

io.configurar_estilo()

SEMENTE = config.SEMENTE

# Varredura de k: começa em 1 porque o peso com um único grupo é a âncora da curva
# (a dispersão total da carteira). A silhueta só existe a partir de k = 2.
K_VALORES = list(range(1, 11))

# A faixa que o negócio consegue operar como linha de produto (config do projeto).
FAIXA_NEGOCIO = (config.K_MINIMO_NEGOCIO, config.K_MAXIMO_NEGOCIO)

## 1 · Dados — a coluna `ano_reforma`

In [ ]:
data_path = Path("../data/processed/imoveis_curados.parquet")
dict_path = Path("../data/processed/dicionario.csv")

df = pd.read_parquet(data_path)
df_dict = pd.read_csv(dict_path, sep=";").assign(coluna=lambda d: d["variavel"])

display(Markdown(
    f"### Conjunto de dados — {len(df):,} imóveis × {df.shape[1]} colunas".replace(",", ".")
))
display(df.head())

In [ ]:
COLUNA = "ano_reforma"

colunas_do_dicionario = ["variavel", "coluna_original", "descricao", "valores"]
display(df_dict.loc[df_dict["coluna"] == COLUNA, colunas_do_dicionario])

anos = df[COLUNA]
contagem = anos.value_counts().sort_index()
construidos_antes = df["ano_construcao"] < anos.min()

figura, eixo = plt.subplots(figsize=(12, 3.8))
eixo.bar(contagem.index, contagem.values, color=config.AZUL_COBALTO, width=0.8)
eixo.bar(anos.min(), contagem.loc[anos.min()], color=config.CORAL, width=0.8,
         label=f"{anos.min()}: piso do registro")
eixo.set_xlabel("ano da última reforma")
eixo.set_ylabel("imóveis")
eixo.set_title("Distribuição de `ano_reforma`", fontweight="bold")
eixo.legend()
plt.show()

display(Markdown(
    f"- de **{anos.min()}** a **{anos.max()}**, **{anos.nunique()}** anos distintos\n"
    f"- **{(anos == anos.min()).mean():.1%}** dos imóveis estão em {anos.min()}\n"
    f"- dos construídos antes de {anos.min()}, "
    f"**{(anos[construidos_antes] == anos.min()).mean():.1%}**"
    f" têm `ano_reforma` = {anos.min()}\n"
    f"- **{(anos == df['ano_construcao']).mean():.1%}** nunca foram reformados "
    "(`ano_reforma` = `ano_construcao`)"
))

**O pico de 1950 é um piso do registro, não uma onda de reformas.** O cadastro de Ames não
anota reforma anterior a 1950: casa mais antiga sem reforma registrada recebe 1950. O valor
quer dizer "1950 **ou antes**". Para a distância isso importa: centenas de imóveis empilhados
no mesmo ponto formam um bloco compacto, e bloco compacto é exatamente o que a silhueta premia.

Quando o imóvel nunca foi reformado, `ano_reforma` é o próprio ano de construção. A variável
mistura as duas coisas: "reformado em 1995" e "construído em 1995 e nunca reformado" caem no
mesmo ponto.

O pré-processamento é um `MinMaxScaler`: 1950 vira 0, 2010 vira 1, e **um ano** vale
`PASSO_ANO` na escala. Em uma dimensão a escala não muda a silhueta — todas as distâncias
encolhem na mesma proporção —, mas mantém a convenção dos outros notebooks.

In [ ]:
X = df[[COLUNA]]  # aprendizagem não supervisionada: não há y
X_T = MinMaxScaler().fit_transform(X)
N = len(X_T)
PASSO_ANO = 1 / (anos.max() - anos.min())

display(Markdown(
    f"Espaço pré-processado: **{N:,} × {X_T.shape[1]}**".replace(",", ".")
    + f" · um ano = **{PASSO_ANO:.4f}** na escala [0, 1]"
))

## 2 · Os três espaços de distância

A silhueta e o K-Medoids trabalham sobre distâncias entre pares de imóveis. Elas são
calculadas **uma vez** e reaproveitadas em toda varredura — é o que permite medir a
silhueta a cada *k* sem que o gráfico demore a andar.

**Em uma dimensão, as três réguas são a mesma.** Euclidiana e Manhattan dão |x − y|; a
de Mahalanobis divide isso pelo desvio-padrão. A célula confere as duas igualdades. Os oito
algoritmos continuam diferentes — muda o **centro** (média, mediana, medoide), o critério de
fusão ou a densidade —, mas a silhueta de todos é medida, na prática, com a mesma régua.

**Mahalanobis.** A distância de Mahalanobis desconta a correlação entre as variáveis:
$d(x, y) = \sqrt{(x-y)^\top \Sigma^{-1} (x-y)}$. Ela é exatamente a distância euclidiana
depois de **branquear** os dados — rotacionar para os componentes principais e dividir cada
um pelo próprio desvio-padrão. Rodar o K-Means euclidiano no espaço branqueado **é** rodar o
K-Means com Mahalanobis, e a célula confere isso numericamente.

Com uma coluna, $\Sigma$ é a variância, um número; o branqueamento só divide pelo
desvio-padrão.

In [ ]:
# Euclidiana e Manhattan, no espaço pré-processado.
D_EUCLIDIANA = squareform(pdist(X_T, "euclidean"))
D_MANHATTAN = squareform(pdist(X_T, "cityblock"))

# Branqueamento: Z = (X - μ) V Λ^(-1/2), só nas direções com variância > 0.
covariancia = np.atleast_2d(np.cov(X_T, rowvar=False))  # 1 × 1 com uma coluna
autovalores, autovetores = np.linalg.eigh(covariancia)
mantidos = autovalores > 1e-10 * autovalores.max()
Z_MAHALANOBIS = (X_T - X_T.mean(axis=0)) @ autovetores[:, mantidos] / np.sqrt(autovalores[mantidos])
D_MAHALANOBIS = squareform(pdist(Z_MAHALANOBIS, "euclidean"))

# A prova: Mahalanobis com a pseudo-inversa = euclidiana no espaço branqueado.
amostra = slice(0, 200)
referencia = cdist(X_T[amostra], X_T[amostra], "mahalanobis", VI=np.linalg.pinv(covariancia))
erro = np.abs(referencia - D_MAHALANOBIS[amostra, amostra]).max()
assert erro < 1e-8, f"o branqueamento não reproduz Mahalanobis (erro {erro:.2e})"

# As igualdades de uma dimensão: Manhattan = euclidiana; Mahalanobis = euclidiana ÷ σ.
assert np.allclose(D_MANHATTAN, D_EUCLIDIANA)
assert np.allclose(D_MAHALANOBIS * np.sqrt(covariancia[0, 0]), D_EUCLIDIANA)

display(Markdown(
    f"- direções com variância nula descartadas no branqueamento: "
    f"**{(~mantidos).sum()}** de {len(mantidos)}\n"
    f"- erro máximo entre Mahalanobis (`scipy`) e euclidiana branqueada: **{erro:.1e}**"
))

## 3 · O gráfico ao vivo

Cada varredura abre um painel com dois gráficos e o **redesenha a cada *k* treinado**: o
peso à esquerda, a silhueta à direita. Quando a varredura termina, o painel marca:

* a **linha tracejada** no cotovelo — o *k* mais distante da corda que liga o primeiro ao
  último ponto da curva de peso (`part.cotovelo`, o mesmo critério do notebook 04);
* a **estrela** no *k* de maior silhueta;
* a **faixa sombreada** de 2 a 5 grupos, o que o negócio opera;
* a **linha pontilhada** na silhueta mínima aceita pelo projeto.

Quando cotovelo e estrela coincidem, as duas leituras concordam. Quando divergem, o peso
está premiando grupos que a silhueta não reconhece como separados — e vale desconfiar.

In [ ]:
RESULTADOS: dict[str, dict] = {}


def contar_grupos(rotulos: np.ndarray) -> int:
    # Grupos válidos: o rótulo -1 é ruído (DBSCAN) e não conta como grupo.
    return len(np.unique(rotulos[rotulos >= 0]))


def soma_quadrados_intra(dados: np.ndarray, rotulos: np.ndarray) -> float:
    # Soma das distâncias quadráticas de cada ponto ao centroide do próprio grupo.
    return float(sum(
        ((dados[rotulos == g] - dados[rotulos == g].mean(axis=0)) ** 2).sum()
        for g in np.unique(rotulos)
    ))


def silhueta_sobre(distancias: np.ndarray):
    # Devolve uma função que mede a silhueta numa matriz de distâncias pré-calculada.
    # Ruído fica de fora; com menos de dois grupos, a silhueta não existe (NaN).
    def medir(rotulos: np.ndarray) -> float:
        validos = rotulos >= 0
        grupos = contar_grupos(rotulos)
        if grupos < 2 or validos.sum() <= grupos:
            return np.nan
        submatriz = distancias[np.ix_(validos, validos)]
        return float(silhouette_score(submatriz, rotulos[validos], metric="precomputed"))

    return medir


silhueta_comum = silhueta_sobre(D_EUCLIDIANA)


def cotovelo_continuo(curva: pd.Series):
    # O critério de `part.cotovelo` (maior distância à corda) para um eixo contínuo,
    # como bandwidth e eps. Devolve o valor do parâmetro, não um k.
    x = curva.index.to_numpy(dtype=float)
    y = curva.to_numpy(dtype=float)
    pontos = np.column_stack([(x - x.min()) / (x.max() - x.min()),
                              (y - y.min()) / (y.max() - y.min())])
    inicio, fim = pontos[0], pontos[-1]
    vetor = (fim - inicio) / np.linalg.norm(fim - inicio)
    relativos = pontos - inicio
    distancias = np.linalg.norm(relativos - np.outer(relativos @ vetor, vetor), axis=1)
    return curva.index[int(np.argmax(distancias))]


class CotoveloAoVivo:
    # Painel peso × silhueta, redesenhado a cada ponto da varredura.

    def __init__(self, nome, rotulo_peso, rotulo_silhueta, rotulo_x, por_parametro):
        self.nome = nome
        self.rotulo_peso = rotulo_peso
        self.rotulo_silhueta = rotulo_silhueta
        self.rotulo_x = rotulo_x
        self.por_parametro = por_parametro
        self.figura, (self.eixo_peso, self.eixo_silhueta) = plt.subplots(1, 2, figsize=(13, 4.3))
        plt.close(self.figura)  # quem exibe é o handle, não o backend inline
        self.saida = display(self.figura, display_id=True)

    def desenhar(self, tabela: pd.DataFrame, rodape: str, cotovelo=None, melhor=None):
        x = tabela.index.to_numpy(dtype=float)
        for eixo in (self.eixo_peso, self.eixo_silhueta):
            eixo.clear()
            eixo.grid(alpha=0.3)
            eixo.set_xlabel(self.rotulo_x)
            if not self.por_parametro:
                eixo.axvspan(*FAIXA_NEGOCIO, color=config.VERDE, alpha=0.12, lw=0)
                eixo.set_xticks(K_VALORES)
                eixo.set_xlim(K_VALORES[0] - 0.3, K_VALORES[-1] + 0.3)
            if cotovelo is not None:
                eixo.axvline(cotovelo, color=config.AZUL_PROFUNDO, ls="--", lw=1.2,
                             label="cotovelo")

        self.eixo_peso.plot(x, tabela["peso"], "o-", color=config.AZUL_COBALTO)
        self.eixo_peso.set_title(f"Peso — {self.rotulo_peso}", fontsize=10)

        self.eixo_silhueta.plot(x, tabela["silhueta"], "o-", color=config.CORAL)
        self.eixo_silhueta.axhline(config.SILHUETA_MINIMA, color="gray", ls=":", lw=1,
                                   label=f"silhueta mínima ({config.SILHUETA_MINIMA})")
        self.eixo_silhueta.set_title(f"Silhueta — {self.rotulo_silhueta}", fontsize=10)
        if melhor is not None:
            self.eixo_silhueta.plot(melhor, tabela.loc[melhor, "silhueta"], "*", ms=18,
                                    color=config.AMARELO, mec="black", label="maior silhueta")

        if self.por_parametro:
            # O eixo vai do maior para o menor parâmetro: k cresce da esquerda para a direita.
            for eixo in (self.eixo_peso, self.eixo_silhueta):
                eixo.invert_xaxis()
            self.eixo_peso.margins(y=0.25)  # espaço para a anotação acima de cada ponto
            ruido = tabela["pct_ruido"] if "pct_ruido" in tabela else [None] * len(tabela)
            for valor, grupos, peso, pct in zip(x, tabela["grupos"], tabela["peso"], ruido):
                rotulo = f"k={grupos}" if pct is None else f"k={grupos}\n{pct:.0%} ruído"
                self.eixo_peso.annotate(rotulo, (valor, peso),
                                        textcoords="offset points", xytext=(0, 7),
                                        ha="center", fontsize=8)

        self.eixo_silhueta.legend(fontsize=8, loc="best")
        self.figura.suptitle(f"{self.nome} · {rodape}", fontsize=12, fontweight="bold")
        self.figura.tight_layout()
        self.saida.update(self.figura)


def varrer(nome, valores, ajustar, medir_silhueta, rotulo_peso, rotulo_silhueta,
           rotulo_x="k — número de grupos", por_parametro=False, preparo=0.0):
    # Treina `ajustar(valor)` para cada valor, mede peso e silhueta e atualiza o painel.
    #
    # `ajustar` devolve um dicionário com `rotulos`, `peso` e, opcionalmente, colunas
    # extras para a tabela. `preparo` soma ao tempo o custo feito uma vez fora do laço.
    painel = CotoveloAoVivo(nome, rotulo_peso, rotulo_silhueta, rotulo_x, por_parametro)
    linhas = []
    for valor in valores:
        inicio = time.perf_counter()
        saida = dict(ajustar(valor))
        segundos = time.perf_counter() - inicio
        rotulos = saida.pop("rotulos")
        validos = rotulos[rotulos >= 0]
        linhas.append({
            "x": valor,
            "grupos": contar_grupos(rotulos),
            "peso": saida.pop("peso"),
            "silhueta": medir_silhueta(rotulos),
            "silhueta_comum": silhueta_comum(rotulos),
            "maior_grupo_pct": np.bincount(validos).max() / N if len(validos) else np.nan,
            "segundos": segundos,
            **saida,
        })
        tabela = pd.DataFrame(linhas).set_index("x")
        painel.desenhar(tabela, f"{rotulo_x.split(' ')[0]} = {valor} treinado em {segundos:.1f}s")

    # O cotovelo só existe numa curva com pelo menos três pontos distintos.
    # `part.cotovelo` devolve um k inteiro; com o eixo no parâmetro (contínuo), a versão
    # contínua do mesmo critério devolve o valor do parâmetro.
    cotovelo = None
    if tabela["peso"].nunique() >= 3:
        medir_cotovelo = cotovelo_continuo if por_parametro else part.cotovelo
        cotovelo = medir_cotovelo(tabela["peso"])
    melhor = tabela["silhueta"].idxmax() if tabela["silhueta"].notna().any() else None
    total = tabela["segundos"].sum() + preparo
    painel.desenhar(tabela, f"varredura concluída em {total:.1f}s", cotovelo, melhor)

    RESULTADOS[nome] = {"tabela": tabela, "cotovelo": cotovelo, "melhor": melhor,
                        "segundos": total, "por_parametro": por_parametro}
    tabela.index.name = rotulo_x.split(" ")[0]
    display(tabela.style.format(precision=4, na_rep="—").format(
        {"peso": "{:,.2f}", "maior_grupo_pct": "{:.1%}", "segundos": "{:.2f}"}
    ))
    return tabela

# 4 · Os algoritmos

## 4.1 · K-Means — distância euclidiana

O K-Means clássico: cada imóvel vai para o centroide mais próximo e cada centroide vira a
média do seu grupo, até estabilizar. O peso é a **inércia**, a soma dos quadrados das
distâncias ao centroide. Dez inicializações `k-means++` por *k*; fica a de menor inércia.

In [ ]:
def ajustar_kmeans_euclidiano(k):
    modelo = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=SEMENTE).fit(X_T)
    return {"rotulos": modelo.labels_, "peso": modelo.inertia_}


kmeans_euclidiano = varrer(
    "K-Means · euclidiana", K_VALORES, ajustar_kmeans_euclidiano,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="inércia (soma dos quadrados)", rotulo_silhueta="euclidiana",
)

## 4.2 · K-Means — distância Manhattan

Troca-se a régua: o imóvel vai para o centro com menor soma de diferenças absolutas
(distância L1). Com essa régua, a média **deixa de ser** o melhor centro — quem minimiza a
soma das distâncias L1 é a **mediana**, coordenada a coordenada. Manter a média faria o
peso subir em algumas iterações e o algoritmo não convergiria. Por isso a literatura chama
esta variante de *K-Medianas*: é o K-Means com a régua Manhattan levada às últimas
consequências.

O `scikit-learn` não traz essa variante; a implementação abaixo é o laço de Lloyd com a
atribuição em L1 e a atualização pela mediana, com inicialização no estilo `k-means++`.
O peso é a soma das distâncias L1 de cada imóvel ao seu centro.

Em uma dimensão a régua L1 é a euclidiana; o que distingue este algoritmo do 4.1 é só o
centro — a **mediana** do grupo, e não a média. Com um pico de imóveis em 1950, a mediana
gruda no pico e a média não.

In [ ]:
def kmeans_manhattan(dados, k, reinicios=10, max_iteracoes=100, semente=SEMENTE):
    # Laço de Lloyd com distância L1: atribuição ao centro mais próximo, centro = mediana.
    gerador = np.random.default_rng(semente)
    n = len(dados)
    melhor = None
    for _ in range(reinicios):
        # Inicialização k-means++ com a régua L1: cada centro novo sorteado longe dos atuais.
        centros = [dados[gerador.integers(n)]]
        for _ in range(1, k):
            proximidade = cdist(dados, np.array(centros), "cityblock").min(axis=1)
            centros.append(dados[gerador.choice(n, p=proximidade / proximidade.sum())])
        centros = np.array(centros)

        for _ in range(max_iteracoes):
            rotulos = cdist(dados, centros, "cityblock").argmin(axis=1)
            novos = np.array([
                np.median(dados[rotulos == g], axis=0) if np.any(rotulos == g) else centros[g]
                for g in range(k)
            ])
            if np.allclose(novos, centros):
                break
            centros = novos

        distancias = cdist(dados, centros, "cityblock")
        rotulos = distancias.argmin(axis=1)
        custo = float(distancias[np.arange(n), rotulos].sum())
        if melhor is None or custo < melhor[1]:
            melhor = (rotulos, custo)
    return melhor


def ajustar_kmeans_manhattan(k):
    rotulos, custo = kmeans_manhattan(X_T, k)
    return {"rotulos": rotulos, "peso": custo}


kmeans_manhattan_tabela = varrer(
    "K-Means · Manhattan", K_VALORES, ajustar_kmeans_manhattan,
    silhueta_sobre(D_MANHATTAN),
    rotulo_peso="soma das distâncias L1", rotulo_silhueta="Manhattan",
)

## 4.3 · K-Means — distância de Mahalanobis

O K-Means euclidiano roda sobre o espaço branqueado do bloco 2, e isso é o K-Means com
Mahalanobis. O peso é a soma dos quadrados das distâncias de Mahalanobis ao centroide, e a
silhueta é medida em Mahalanobis.

Em uma dimensão o branqueamento só divide a coluna pelo desvio-padrão, e o K-Means não
muda com a escala: os grupos devem sair **iguais** aos do 4.1, e a silhueta de Mahalanobis
igual à euclidiana. A varredura fica aqui para mostrar isso, não para competir.

In [ ]:
def ajustar_kmeans_mahalanobis(k):
    modelo = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=SEMENTE)
    modelo.fit(Z_MAHALANOBIS)
    return {"rotulos": modelo.labels_, "peso": modelo.inertia_}


kmeans_mahalanobis = varrer(
    "K-Means · Mahalanobis", K_VALORES, ajustar_kmeans_mahalanobis,
    silhueta_sobre(D_MAHALANOBIS),
    rotulo_peso="soma dos quadrados de Mahalanobis", rotulo_silhueta="Mahalanobis",
)

## 4.4 · K-Medoids

O centro do grupo deixa de ser uma média e passa a ser um **imóvel real da carteira**, o
medoide: o que tem a menor soma de distâncias aos demais do grupo. É a implementação do
projeto (`part.kmedoids`, iteração de Voronoi com inicialização `k-means++`), sobre a
matriz euclidiana — a mesma régua do 4.1, para que a diferença entre os dois venha só do
centro. O peso é a soma das distâncias (não ao quadrado) de cada imóvel ao seu medoide.

In [ ]:
def ajustar_kmedoids(k):
    resultado = part.kmedoids(D_EUCLIDIANA, k, reinicios=5, semente=SEMENTE)
    return {"rotulos": resultado.rotulos, "peso": resultado.custo}


kmedoids = varrer(
    "K-Medoids", K_VALORES, ajustar_kmedoids,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma das distâncias ao medoide", rotulo_silhueta="euclidiana",
)

## 4.5 · Modelos de Mistura Gaussiana (GMM)

O GMM supõe que a carteira é a soma de *k* gaussianas e estima, para cada imóvel, a
probabilidade de pertencer a cada uma. O rótulo é a componente mais provável. O peso é a
**−log-verossimilhança** total: quanto menor, melhor a mistura explica os dados — e, como a
inércia, ela cai a cada componente acrescentada. A tabela traz também o **BIC**, que
penaliza o número de parâmetros e por isso **não** cai sempre: é o contraponto do próprio
GMM ao cotovelo.

Em uma dimensão, cada componente é uma curva normal sobre os anos: a mistura procura *k*
"sinos" que, somados, reproduzam o histograma do bloco 1 — inclusive o pico de 1950.

In [ ]:
def ajustar_gmm(k):
    modelo = GaussianMixture(
        n_components=k, covariance_type="full",
        max_iter=200, random_state=SEMENTE,
    ).fit(X_T)
    return {
        "rotulos": modelo.predict(X_T),
        "peso": -modelo.score(X_T) * N,
        "bic": modelo.bic(X_T),
    }


gmm = varrer(
    "Mistura gaussiana (GMM)", K_VALORES, ajustar_gmm,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="−log-verossimilhança", rotulo_silhueta="euclidiana",
)

## 4.6 · Clusterização hierárquica (Ward)

A aglomerativa começa com cada imóvel sozinho e funde, a cada passo, o par de grupos que
menos aumenta a soma dos quadrados intra-grupo (critério de Ward). O dendrograma é
construído **uma vez**; cada *k* é um corte nele (`hier.cortar`). O tempo de construção
entra no total da varredura.

O peso é a soma dos quadrados intra-grupo — exatamente o que Ward minimiza a cada fusão.

In [ ]:
inicio = time.perf_counter()
ligacao_ward = hier.matriz_ligacao(X_T, ligacao="ward")
preparo_ward = time.perf_counter() - inicio


def ajustar_hierarquica(k):
    rotulos = hier.cortar(ligacao_ward, k)
    return {"rotulos": rotulos, "peso": soma_quadrados_intra(X_T, rotulos)}


hierarquica = varrer(
    "Hierárquica (Ward)", K_VALORES, ajustar_hierarquica,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma dos quadrados intra-grupo", rotulo_silhueta="euclidiana",
    preparo=preparo_ward,
)

## 4.7 · Mean Shift

O Mean Shift desloca uma janela de raio `bandwidth` em direção à região mais densa até
parar num **modo** da densidade; imóveis que param no mesmo modo formam um grupo. O número
de grupos **emerge** da largura da janela: janela larga, poucos modos; janela estreita,
muitos.

A varredura percorre larguras estimadas por quantis da distância entre vizinhos
(`estimate_bandwidth`), da maior para a menor — então *k* cresce da esquerda para a direita
e cada ponto do peso traz o *k* que resultou. O peso é a soma dos quadrados intra-grupo.

Em uma dimensão o `bin_seeding=True` funciona — as sementes são uma grade sobre os anos —
e deixa o ajuste rápido. Larguras nulas (quantis pequenos sobre anos repetidos) ficam fora.

In [ ]:
QUANTIS_MEAN_SHIFT = (0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.07, 0.05)
larguras = sorted(
    {round(float(estimate_bandwidth(X_T, quantile=q, n_samples=1000, random_state=SEMENTE)), 3)
     for q in QUANTIS_MEAN_SHIFT} - {0.0},
    reverse=True,
)


def ajustar_mean_shift(largura):
    modelo = MeanShift(bandwidth=largura, bin_seeding=True).fit(X_T)
    return {"rotulos": modelo.labels_, "peso": soma_quadrados_intra(X_T, modelo.labels_)}


mean_shift = varrer(
    "Mean Shift", larguras, ajustar_mean_shift,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma dos quadrados intra-grupo", rotulo_silhueta="euclidiana",
    rotulo_x="bandwidth — largura da janela", por_parametro=True,
)

## 4.8 · DBSCAN

O DBSCAN junta imóveis que têm pelo menos `min_samples` vizinhos num raio `eps`; o que não
alcança densidade suficiente vira **ruído** (rótulo −1). Como no Mean Shift, *k* emerge do
parâmetro.

O DBSCAN tem um cotovelo próprio, anterior à varredura: a **curva k-distância** — a
distância de cada imóvel ao seu `min_samples`-ésimo vizinho, ordenada. O joelho dela é o
`eps` a partir do qual os pontos deixam de ter vizinhos próximos. Com anos repetidos, a
curva fica em degraus: quase todo imóvel tem dez vizinhos **no mesmo ano**, a distância zero.

Por isso a varredura de `eps` não usa percentis da curva — a maioria seria zero —, e sim
**anos**: de 8 anos até meio ano, do maior para o menor raio. O que esperar: com raio de um
ano ou mais, anos vizinhos se encadeiam e a carteira inteira vira um grupo; abaixo de um ano,
cada ano com ao menos `min_samples` imóveis vira o seu próprio grupo. Em uma dimensão densa
como esta, o DBSCAN não tem meio-termo.

O peso precisa decidir o que fazer com o ruído. Ignorá-lo faria o peso cair só porque os
imóveis estão sendo **descartados**, não agrupados. Aqui o ruído paga o custo de quem não
tem grupo: a distância ao centro da carteira inteira, a mesma que pagaria com *k* = 1. A
silhueta é medida sem o ruído, e a coluna `pct_ruido` mostra quanto ficou de fora.

In [ ]:
MIN_AMOSTRAS = 10

curva = den.curva_k_distancia(X_T, MIN_AMOSTRAS)
indice_joelho, eps_joelho = den.joelho_k_distancia(curva)

figura, eixo = plt.subplots(figsize=(8, 4))
eixo.plot(curva, color=config.AZUL_COBALTO)
eixo.axhline(eps_joelho, color=config.CORAL, ls="--", label=f"joelho: eps = {eps_joelho:.2f}")
eixo.set_xlabel("imóveis, ordenados")
eixo.set_ylabel(f"distância ao {MIN_AMOSTRAS}º vizinho")
eixo.set_title("DBSCAN · curva k-distância", fontweight="bold")
eixo.grid(alpha=0.3)
eixo.legend()
plt.show()

In [ ]:
ANOS_EPS = (8, 5, 3, 2, 1.5, 1.01, 0.75, 0.5)  # 1.01: um ano, com folga de arredondamento
valores_eps = [round(anos_eps * PASSO_ANO, 4) for anos_eps in ANOS_EPS]
centro_da_carteira = X_T.mean(axis=0)


def ajustar_dbscan(eps):
    rotulos = DBSCAN(eps=eps, min_samples=MIN_AMOSTRAS).fit_predict(X_T)
    ruido = rotulos < 0
    # Agrupados pagam a distância ao centro do grupo; ruído, ao centro da carteira.
    peso = soma_quadrados_intra(X_T[~ruido], rotulos[~ruido]) if (~ruido).any() else 0.0
    peso += float(((X_T[ruido] - centro_da_carteira) ** 2).sum())
    return {"rotulos": rotulos, "peso": peso, "pct_ruido": ruido.mean()}


dbscan = varrer(
    "DBSCAN", valores_eps, ajustar_dbscan,
    silhueta_sobre(D_EUCLIDIANA),
    rotulo_peso="soma dos quadrados, ruído sem grupo",
    rotulo_silhueta="euclidiana, sem ruído",
    rotulo_x="eps — raio de vizinhança", por_parametro=True,
)

# 5 · O comparativo

Para cada algoritmo, dois candidatos a *k*: o do **cotovelo** do peso e o da **maior
silhueta**. A silhueta de cada um está na régua do próprio algoritmo; a **silhueta comum**
é a euclidiana, a mesma para todos, e é ela que permite ler a tabela na vertical.

Duas colunas protegem a leitura: `maior grupo` e `ruído`. Uma silhueta alta com um grupo
que concentra quase toda a carteira não é segmentação — é a carteira inteira mais um punhado
de exceções.

In [ ]:
def linha_do_resumo(nome, r):
    tabela, cotovelo, melhor = r["tabela"], r["cotovelo"], r["melhor"]

    def em(x, coluna):
        return tabela.loc[x, coluna] if x is not None else np.nan

    na_faixa = tabela[tabela["grupos"].between(*FAIXA_NEGOCIO)]["silhueta"].dropna()
    melhor_faixa = na_faixa.idxmax() if len(na_faixa) else None
    return {
        "algoritmo": nome,
        "k no cotovelo": em(cotovelo, "grupos"),
        "silhueta no cotovelo": em(cotovelo, "silhueta"),
        "k de maior silhueta": em(melhor, "grupos"),
        "maior silhueta": em(melhor, "silhueta"),
        "silhueta comum": em(melhor, "silhueta_comum"),
        "maior grupo": em(melhor, "maior_grupo_pct"),
        "ruído": em(melhor, "pct_ruido") if "pct_ruido" in tabela else 0.0,
        "k de maior silhueta em 2–5": em(melhor_faixa, "grupos"),
        "silhueta comum em 2–5": em(melhor_faixa, "silhueta_comum"),
        "tempo (s)": r["segundos"],
    }


resumo = pd.DataFrame([linha_do_resumo(n, r) for n, r in RESULTADOS.items()]).set_index("algoritmo")
for coluna in ("k no cotovelo", "k de maior silhueta", "k de maior silhueta em 2–5"):
    resumo[coluna] = resumo[coluna].astype("Int64")

display(
    resumo.style
    .format(precision=3, na_rep="—")
    .format({"maior grupo": "{:.1%}", "ruído": "{:.1%}", "tempo (s)": "{:.1f}"}, na_rep="—")
    .highlight_max(subset=["silhueta comum", "silhueta comum em 2–5"], color="#d8f3dc")
)

In [ ]:
figura, (eixo_peso, eixo_silhueta) = plt.subplots(1, 2, figsize=(14, 5))
cores = plt.cm.tab10.colors

for cor, (nome, r) in zip(cores, RESULTADOS.items()):
    tabela = r["tabela"].sort_values("grupos")
    estilo = "s--" if r["por_parametro"] else "o-"
    # Peso normalizado: 1 no maior peso de cada algoritmo, 0 no menor. Os pesos têm
    # unidades diferentes (e a −log-verossimilhança do GMM pode ser negativa).
    peso = tabela["peso"]
    eixo_peso.plot(tabela["grupos"], (peso - peso.min()) / (peso.max() - peso.min()), estilo,
                   color=cor, label=nome, alpha=0.85)
    eixo_silhueta.plot(tabela["grupos"], tabela["silhueta_comum"], estilo,
                       color=cor, label=nome, alpha=0.85)

for eixo in (eixo_peso, eixo_silhueta):
    eixo.axvspan(*FAIXA_NEGOCIO, color=config.VERDE, alpha=0.12, lw=0)
    eixo.set_xlabel("k — grupos obtidos")
    eixo.set_xlim(0.5, K_VALORES[-1] + 0.5)
    eixo.grid(alpha=0.3)
eixo_silhueta.axhline(config.SILHUETA_MINIMA, color="gray", ls=":", lw=1)
eixo_peso.set_title("Peso normalizado (1 = maior, 0 = menor de cada algoritmo)", fontsize=10)
eixo_silhueta.set_title("Silhueta comum (euclidiana, mesmo espaço)", fontsize=10)
eixo_silhueta.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
figura.suptitle("Todos os algoritmos na mesma régua", fontsize=12, fontweight="bold")
figura.tight_layout()
plt.show()

## 6 · A leitura

A regra para escolher, declarada antes de olhar o resultado:

1. o *k* precisa estar na faixa que o negócio opera (2 a 5 grupos);
2. o ruído não pode passar do teto do projeto, e nenhum grupo pode concentrar mais de 90%
   da carteira — acima disso a partição não segmenta, separa exceções;
3. entre os que sobram, vence a maior **silhueta comum**; abaixo da silhueta mínima do
   projeto, o vencedor é registrado com a ressalva de que a estrutura é fraca.

A célula abaixo aplica a regra sobre as varreduras e diz o que sobrou.

In [ ]:
GRUPO_DOMINANTE_MAXIMO = 0.90

candidatos = pd.concat(
    [r["tabela"].assign(algoritmo=nome).reset_index(drop=True) for nome, r in RESULTADOS.items()],
    ignore_index=True,
)
if "pct_ruido" not in candidatos:
    candidatos["pct_ruido"] = np.nan
candidatos["pct_ruido"] = candidatos["pct_ruido"].fillna(0.0)

aprovados = candidatos[
    candidatos["grupos"].between(*FAIXA_NEGOCIO)
    & (candidatos["pct_ruido"] <= config.RUIDO_MAXIMO)
    & (candidatos["maior_grupo_pct"] <= GRUPO_DOMINANTE_MAXIMO)
    & candidatos["silhueta_comum"].notna()
].sort_values("silhueta_comum", ascending=False)

reprovados = sorted(set(RESULTADOS) - set(aprovados["algoritmo"]))
display(Markdown(
    f"**{len(aprovados)}** configurações passam na regra, de **{len(candidatos)}** treinadas. "
    + (f"Sem nenhuma aprovada: {', '.join(reprovados)}." if reprovados else "")
))

if len(aprovados):
    display(
        aprovados[["algoritmo", "grupos", "silhueta", "silhueta_comum", "maior_grupo_pct",
                   "pct_ruido"]]
        .head(10).style.hide(axis="index")
        .format(precision=3).format({"maior_grupo_pct": "{:.1%}", "pct_ruido": "{:.1%}"})
    )
    vencedor = aprovados.iloc[0]
    ressalva = (
        "" if vencedor["silhueta_comum"] >= config.SILHUETA_MINIMA
        else f" — **abaixo da silhueta mínima do projeto ({config.SILHUETA_MINIMA})**: "
        "a estrutura existe, mas é fraca"
    )
    display(Markdown(
        f"Maior silhueta comum dentro da regra: **{vencedor['algoritmo']}** com "
        f"**k = {vencedor['grupos']}** (silhueta comum **{vencedor['silhueta_comum']:.3f}**)"
        f"{ressalva}."
    ))

## A meta: silhueta entre 0,7 e 0,9

A regra do bloco 6 pede uma partição que o negócio opere. A meta deste notebook é mais
estreita: quais configurações, entre as que passam na regra, chegam à faixa pedida.

In [ ]:
META_SILHUETA = (0.7, 0.9)

na_meta = aprovados[aprovados["silhueta_comum"].between(*META_SILHUETA)]
display(Markdown(
    f"**{len(na_meta)}** configurações aprovadas atingem silhueta entre "
    f"**{META_SILHUETA[0]}** e **{META_SILHUETA[1]}**."
))
if len(na_meta):
    display(
        na_meta[["algoritmo", "grupos", "silhueta_comum", "maior_grupo_pct"]]
        .style.hide(axis="index")
        .format(precision=3).format({"maior_grupo_pct": "{:.1%}"})
    )
fora_da_meta = aprovados[aprovados["silhueta_comum"] < META_SILHUETA[0]]
if len(fora_da_meta):
    primeira_fora = fora_da_meta.iloc[0]  # `aprovados` já vem em ordem decrescente
    display(Markdown(
        f"Abaixo da meta, a maior silhueta aprovada é **{primeira_fora['silhueta_comum']:.3f}** "
        f"({primeira_fora['algoritmo']}, k = {primeira_fora['grupos']})."
    ))

# 7 · O que são os grupos

Em uma dimensão, cada grupo é um intervalo de anos. A tabela mostra os intervalos do K-Means
em cada *k* da faixa de negócio, e a silhueta de um corte **sem algoritmo**: dividir os anos
em *k* faixas com o mesmo número de imóveis (`pd.qcut`). Se o corte por quantis chega perto,
o algoritmo está fazendo pouco mais que fatiar a reta.

In [ ]:
linhas = []
for k in range(FAIXA_NEGOCIO[0], FAIXA_NEGOCIO[1] + 1):
    rotulos = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit_predict(X_T)
    faixas = (
        df.assign(grupo=rotulos).groupby("grupo")[COLUNA]
        .agg(["min", "max", "size"]).sort_values("min")
    )
    corte = pd.qcut(anos.rank(method="first"), k, labels=False).to_numpy()
    linhas.append({
        "k": k,
        "faixas do K-Means": " · ".join(f"{a}–{b} ({n})" for a, b, n in faixas.to_numpy()),
        "silhueta K-Means": silhouette_score(X_T, rotulos),
        "silhueta do corte por quantis": silhouette_score(X_T, corte),
    })

display(
    pd.DataFrame(linhas).set_index("k").style
    .format({"silhueta K-Means": "{:.3f}", "silhueta do corte por quantis": "{:.3f}"})
)

### O perfil de cada faixa

O agrupamento usou só o ano da reforma; as outras colunas entram **depois**, para descrever o
que cada faixa reúne. O *k* é o de maior silhueta do K-Means na faixa de negócio.

In [ ]:
tabela_km = RESULTADOS["K-Means · euclidiana"]["tabela"]
k_final = int(tabela_km.loc[FAIXA_NEGOCIO[0]:FAIXA_NEGOCIO[1], "silhueta"].idxmax())
rotulos_finais = KMeans(n_clusters=k_final, n_init=10, random_state=SEMENTE).fit_predict(X_T)

# Grupos numerados em ordem cronológica: 0 é a faixa mais antiga.
ordem = df.assign(g=rotulos_finais).groupby("g")[COLUNA].min().sort_values().index
rotulos_finais = pd.Series(rotulos_finais).map({g: i for i, g in enumerate(ordem)}).to_numpy()

perfil = (
    df.assign(segmento=rotulos_finais, nunca_reformado=anos.eq(df["ano_construcao"]))
    .groupby("segmento")
    .agg(
        reforma_de=(COLUNA, "min"),
        reforma_ate=(COLUNA, "max"),
        imoveis=(COLUNA, "size"),
        nunca_reformados=("nunca_reformado", "mean"),
        construcao_mediana=("ano_construcao", "median"),
        area_habitavel=("area_habitavel", "median"),
        qualidade_geral=("qualidade_geral", "median"),
        preco_mediano=("preco_venda", "median"),
    )
)
perfil["participacao"] = perfil["imoveis"] / N

display(Markdown(
    f"K-Means com **k = {k_final}** — silhueta **{silhouette_score(X_T, rotulos_finais):.3f}**"
))
display(perfil.style.format({
    "nunca_reformados": "{:.0%}", "participacao": "{:.1%}", "construcao_mediana": "{:.0f}",
    "area_habitavel": "{:,.0f}", "qualidade_geral": "{:.0f}", "preco_mediano": "${:,.0f}",
}))

figura, eixo = plt.subplots(figsize=(12, 3.8))
cores_faixa = [config.AZUL_PROFUNDO, config.AZUL_CLARO, config.VERDE, config.AMARELO, config.CORAL]
for segmento in range(k_final):
    contagem_seg = anos[rotulos_finais == segmento].value_counts().sort_index()
    eixo.bar(contagem_seg.index, contagem_seg.values, width=0.8,
             color=cores_faixa[segmento % len(cores_faixa)], label=f"segmento {segmento}")
eixo.set_xlabel("ano da última reforma")
eixo.set_ylabel("imóveis")
eixo.set_title(f"Os {k_final} segmentos são faixas de ano", fontweight="bold")
eixo.legend()
plt.show()

## Como ler o que saiu

* **A silhueta chega a 0,7 porque o problema ficou com uma dimensão.** Não há outra variável
  para aproximar dois imóveis de faixas diferentes, e o pico de 1950 forma um bloco compacto
  que a silhueta premia. Com mais grupos, os cortes passam a cair no meio de anos densos e a
  silhueta desce — confira os painéis do bloco 4.
* **Os algoritmos concordam porque, em uma dimensão, quase não há o que discordar.** As três
  réguas coincidem, e "cortar a reta em *k* pedaços" tem pouca liberdade. O DBSCAN é a
  exceção: ou encadeia tudo, ou separa ano a ano.
* **O DBSCAN mostra o limite da métrica.** Separando ano a ano, ele chega a silhueta **1,0**:
  dentro de cada grupo a distância é zero. São quase sessenta "segmentos", um por ano — a
  silhueta máxima possível, e nenhuma segmentação. A regra do bloco 6 o descarta pelo número
  de grupos, não pela silhueta.
* **O corte por quantis do bloco 7 é a prova dos nove.** Se ele chega perto da silhueta do
  K-Means, a segmentação poderia ter sido escrita como uma regra de negócio — "reformados até
  tal ano" —, sem modelo.
* **O que a segmentação diz e o que não diz.** Ela separa imóveis pela recência da última
  intervenção, e o perfil mostra que isso arrasta idade, padrão e preço. Ela **não** diz nada
  sobre área, bairro ou acabamento além do que vem junto com o ano. A silhueta alta mede a
  nitidez dos cortes na reta dos anos — não a qualidade da segmentação como um todo.
* **Ler a silhueta com a régua certa.** 0,7 aqui e 0,13 no 04.2 não querem dizer que este
  modelo é cinco vezes melhor: são distâncias diferentes, sobre informações diferentes.